# Phase 4 — decoder-only CJPE with AdaLoRA


In [1]:
# Run once, then restart the kernel if HF/PEFT/Transformers were already imported.
%pip install -q --upgrade "huggingface_hub==2.2.0" "transformers==5.20.0" "peft==0.21.2" tokenizers scikit-learn tqdm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 839.9/839.9 kB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 109.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 83.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 94.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 3.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 2.2.0 which is incompatible.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggin

In [2]:
import os, json, math, random, gc, tempfile, zipfile, uuid, re, hashlib
from pathlib import Path
from collections import Counter
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tokenizers import Tokenizer
from huggingface_hub import hf_hub_download, HfFileSystem
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix
from tqdm.auto import tqdm

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SEED = 42
DATA_REPO = 'Exploration-Lab/CS-781-Capstone'
BUCKET_ROOT = 'hf://buckets/A-240031/legal-lm-phase1-bucket'
PHASE1_TAG = 'decoder_126M'
MODEL_PATH = 'checkpoints/decoder_126M/model.pt'
TOKENIZER_PATH = 'tokenizer_126M_decoder/tokenizer.json'
MODEL_CONFIG = dict(d_model=1024, n_heads=8, n_kv_heads=4, n_layers=8, d_ff=2816, max_seq_len=2048)
MAX_SEQ_LEN = 2048
CHUNK_OVERLAP = 64  # number of shared tokens, NOT the stride
CHUNK_ENCODE_BATCH_SIZE = 2  # chunks per backbone call
USE_GRADIENT_CHECKPOINTING = False
UNFREEZE_LAST_N_BLOCKS = 0  # frozen backbone; increase only for partial full fine-tuning
TRAIN_FRACTION = 0.20
ADAPT_LAST_N_BLOCKS = 2
RANK_SCHEDULE_EPOCHS = 3
TRAIN_MAX_CHUNKS = 2
EVAL_MAX_CHUNKS = 4
USE_AMP = DEVICE == 'cuda'
AMP_DTYPE = torch.bfloat16 if USE_AMP and torch.cuda.is_bf16_supported() else torch.float16

def autocast_context():
    return torch.autocast(device_type=DEVICE, dtype=AMP_DTYPE, enabled=USE_AMP)

DATA_PREFIX = 'Phase_4_CJPE'
LABEL_KEY = 'label'  # set to the actual binary label field if the released schema differs
assert 0 < MAX_SEQ_LEN <= MODEL_CONFIG['max_seq_len']
assert 0 <= CHUNK_OVERLAP < MAX_SEQ_LEN
assert CHUNK_ENCODE_BATCH_SIZE > 0
assert 0 <= UNFREEZE_LAST_N_BLOCKS <= MODEL_CONFIG['n_layers']
BATCH_SIZE = 1  # one record, at most two training chunks per backward
GRAD_ACCUM_STEPS = 16
LR = 2e-4
WEIGHT_DECAY = 0.01
GRAD_CLIP = 1.0
NUM_EPOCHS = 7
PATIENCE = 5
MIN_DELTA = 0.001
ADAPTER_CONFIG = dict(init_r=16, target_r=8, lora_alpha=32, lora_dropout=0.05,
                      beta1=0.85, beta2=0.85, orth_reg_weight=0.1, deltaT=10)
RANK_WARMUP_FRACTION = 0.10
RANK_FINAL_FRACTION = 0.20
HEAD_DROPOUT = 0.1
UPLOAD_CHECKPOINTS = True
TECHNIQUE = 'adalora'

# All loading/saving locations are controlled here.
# Paths below are relative to BUCKET_ROOT; an absolute hf:// path also works.
SAVE_TAG = None  # None generates a descriptive tag; or enter your own run tag.
SUBMISSION_ZIP_NAME = None  # None uses SAVE_TAG + '.zip'; or enter a filename.
CHECKPOINT_PREFIX = 'checkpoints'
RESUME = False  # True resumes SAVE_TAG; set SAVE_TAG to the existing latest tag.
WORK_ROOT = Path('/kaggle/working/rr_runs') if Path('/kaggle/working').exists() else Path.cwd() / 'cjpe_runs'

if RESUME and SAVE_TAG is None:
    raise ValueError('To resume, set SAVE_TAG to the existing run tag.')
if SAVE_TAG is None:
    SAVE_TAG = f'cjpe_{PHASE1_TAG}_{TECHNIQUE}_len{MAX_SEQ_LEN}_ov{CHUNK_OVERLAP}_sample{TRAIN_MAX_CHUNKS}_eval{EVAL_MAX_CHUNKS}_data{int(TRAIN_FRACTION*100)}_last{ADAPT_LAST_N_BLOCKS}'
if not isinstance(SAVE_TAG, str) or not re.fullmatch(r'[A-Za-z0-9_.-]+', SAVE_TAG) or SAVE_TAG in {'.', '..'}:
    raise ValueError('SAVE_TAG must be a single name containing letters, numbers, _, - or .')
BEST_TAG = SAVE_TAG + '_best'
if SUBMISSION_ZIP_NAME is None:
    SUBMISSION_ZIP_NAME = SAVE_TAG + '.zip'
if Path(SUBMISSION_ZIP_NAME).name != SUBMISSION_ZIP_NAME or not SUBMISSION_ZIP_NAME.endswith('.zip'):
    raise ValueError('SUBMISSION_ZIP_NAME must be a filename ending in .zip.')
WORK_DIR = WORK_ROOT / SAVE_TAG
WORK_DIR.mkdir(parents=True, exist_ok=True)

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret('HF')
    except Exception:
        pass
fs = HfFileSystem(token=HF_TOKEN)

def seed_everything(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
seed_everything(SEED)
print('Device:', DEVICE)
def bucket_path(path):
    if path.startswith('hf://'):
        return path
    return BUCKET_ROOT.rstrip('/') + '/' + path.lstrip('/')

def checkpoint_remote_path(tag):
    return bucket_path(f"{CHECKPOINT_PREFIX.strip('/')}/{tag}/checkpoint.pt")

def download_bucket_file(remote_path):
    # Identity-specific cache directory; refresh the bytes on each call.
    remote = bucket_path(remote_path)
    cache_key = hashlib.sha256(remote.encode()).hexdigest()[:20]
    local = WORK_ROOT / '_inputs' / cache_key / remote.rsplit('/', 1)[-1]
    local.parent.mkdir(parents=True, exist_ok=True)
    temporary = local.with_name(local.name + '.download')
    fs.get_file(remote, str(temporary))
    os.replace(temporary, local)
    return local

def assert_run_destination():
    if RESUME:
        return
    for tag in [SAVE_TAG, BEST_TAG]:
        local = WORK_DIR / 'checkpoints' / tag / 'checkpoint.pt'
        if local.exists() or (UPLOAD_CHECKPOINTS and fs.exists(checkpoint_remote_path(tag))):
            raise FileExistsError(
                f'Checkpoint already exists for {tag}. Choose a new SAVE_TAG, '
                'or set RESUME=True with the original latest SAVE_TAG.'
            )
    manifest = dict(save_tag=SAVE_TAG, best_tag=BEST_TAG, technique=TECHNIQUE,
                    phase1_tag=PHASE1_TAG, model_path=MODEL_PATH,
                    tokenizer_path=TOKENIZER_PATH, submission_zip=SUBMISSION_ZIP_NAME,
                    train_fraction=TRAIN_FRACTION, adapt_last_n_blocks=ADAPT_LAST_N_BLOCKS,
                    train_max_chunks=TRAIN_MAX_CHUNKS, eval_max_chunks=EVAL_MAX_CHUNKS,
                    use_amp=USE_AMP, amp_dtype=str(AMP_DTYPE),
                    max_seq_len=MAX_SEQ_LEN, chunk_overlap=CHUNK_OVERLAP, input_format='overlapping_chunks_v1',
                    pooling='token_mean_then_chunk_mean',
                    unfreeze_last_n_blocks=UNFREEZE_LAST_N_BLOCKS)
    with open(WORK_DIR / 'run_config.json', 'w') as f:
        json.dump(manifest, f, indent=2)

print('Phase-1 model:', bucket_path(MODEL_PATH))
print('Tokenizer:', bucket_path(TOKENIZER_PATH))
print('Latest checkpoint:', checkpoint_remote_path(SAVE_TAG))
print('Best checkpoint:', checkpoint_remote_path(BEST_TAG))
print('Submission ZIP:', WORK_DIR / SUBMISSION_ZIP_NAME)

print('Chunk length/overlap:', MAX_SEQ_LEN, CHUNK_OVERLAP)


Device: cuda
Phase-1 model: hf://buckets/A-240031/legal-lm-phase1-bucket/checkpoints/decoder_126M/model.pt
Tokenizer: hf://buckets/A-240031/legal-lm-phase1-bucket/tokenizer_126M_decoder/tokenizer.json
Latest checkpoint: hf://buckets/A-240031/legal-lm-phase1-bucket/checkpoints/cjpe_decoder_126M_adalora_len2048_ov64_sample2_eval4_data20_last2/checkpoint.pt
Best checkpoint: hf://buckets/A-240031/legal-lm-phase1-bucket/checkpoints/cjpe_decoder_126M_adalora_len2048_ov64_sample2_eval4_data20_last2_best/checkpoint.pt
Submission ZIP: /kaggle/working/rr_runs/cjpe_decoder_126M_adalora_len2048_ov64_sample2_eval4_data20_last2/cjpe_decoder_126M_adalora_len2048_ov64_sample2_eval4_data20_last2.zip
Chunk length/overlap: 2048 64


In [3]:
local_tokenizer = download_bucket_file(TOKENIZER_PATH)
tokenizer = Tokenizer.from_file(str(local_tokenizer))
tokenizer.no_padding()
tokenizer.no_truncation()
VOCAB_SIZE = tokenizer.get_vocab_size()
EMPTY_ID = tokenizer.token_to_id('<|eos|>')
PAD_ID = tokenizer.token_to_id('<|pad|>')
if EMPTY_ID is None:
    raise ValueError('Expected the Phase-1 <|eos|> token.')
if PAD_ID is None: PAD_ID = EMPTY_ID
LABELS = [0, 1]

def download_json(filename):
    path = hf_hub_download(repo_id=DATA_REPO, filename=filename, repo_type='dataset', token=HF_TOKEN)
    with open(path, encoding='utf-8') as f: return json.load(f)

def normalize_documents(raw, labelled):
    if isinstance(raw, list): docs = raw
    elif isinstance(raw, dict) and 'id' in raw and 'text' in raw: docs = [raw]
    elif isinstance(raw, dict):
        docs = []
        for doc_id, value in raw.items():
            if not isinstance(value, dict): raise ValueError('Expected ID -> case object mapping')
            doc = dict(value)
            doc.setdefault('id', doc_id)
            if doc['id'] != doc_id: raise ValueError('Document ID mismatch')
            docs.append(doc)
    else: raise ValueError('Expected a list of cases or ID -> case mapping')
    result = []
    for row_index, doc in enumerate(docs):
        case_id, text = doc['id'], doc['text']
        if not isinstance(text, str): raise ValueError(f'{case_id}: text must be a string')
        row = dict(id=case_id, text=text, row_index=row_index)
        if labelled:
            if LABEL_KEY not in doc:
                raise ValueError(f'Missing label field {LABEL_KEY!r}; available fields: {list(doc)}. Set LABEL_KEY.')
            label = doc[LABEL_KEY]
            if type(label) is not int or label not in (0, 1):
                raise ValueError(f'{case_id}: expected integer label 0/1, got {label!r}')
            row['label'] = label
        result.append(row)
    if not result: raise ValueError('Empty split')
    return result

# Keep each input record separate, even when IDs repeat.
# Test is loaded later, only for prediction.
data = {}
for split in ['train', 'dev']:
    data[split] = normalize_documents(
        download_json(f'{DATA_PREFIX}/Train/{split}.json'), labelled=True)
    counts = Counter(d['id'] for d in data[split])
    print(split, '| records:', len(data[split]), '| unique IDs:', len(counts),
          '| repeated IDs:', sum(n > 1 for n in counts.values()))
    print('Class counts:', dict(Counter(d['label'] for d in data[split])))

# Subset by case ID; repeated records remain separate examples.
# Reloading this cell always starts from the complete downloaded split.
full_train = data['train']
case_labels = {}
for doc in full_train:
    case_labels.setdefault(doc['id'], doc['label'])
    if case_labels[doc['id']] != doc['label']:
        raise ValueError(f"Conflicting labels for training ID {doc['id']}")
case_ids = list(case_labels)
if not 0 < TRAIN_FRACTION <= 1:
    raise ValueError('TRAIN_FRACTION must be in (0, 1]')
if TRAIN_FRACTION < 1:
    chosen, _ = train_test_split(
        case_ids, train_size=TRAIN_FRACTION, random_state=SEED,
        stratify=[case_labels[k] for k in case_ids])
    chosen = set(chosen)
    data['train'] = [doc for doc in full_train if doc['id'] in chosen]
SUBSET_DIGEST = hashlib.sha256(
    json.dumps([d['row_index'] for d in data['train']]).encode()).hexdigest()
DATA_DIGEST = hashlib.sha256(json.dumps(
    {split: docs for split, docs in data.items()},
    sort_keys=True, ensure_ascii=False).encode()).hexdigest()
with open(WORK_DIR / 'train_subset.json', 'w') as f:
    json.dump(dict(fraction=TRAIN_FRACTION, seed=SEED, digest=SUBSET_DIGEST,
                   row_indices=[d['row_index'] for d in data['train']],
                   ids=[d['id'] for d in data['train']]), f)
print('Selected training records:', len(full_train), '→', len(data['train']))
print('Subset classes:', dict(Counter(d['label'] for d in data['train'])))
del full_train


hf://buckets/A-240031/legal-lm-phase1-bu(…): reconstructing file:   0%|          |  0.00B / 2.47MB            

hf://buckets/A-240031/legal-lm-phase1-bu(…): downloading bytes:           |  0.00B            

Phase_4_CJPE/Train/train.json: reconstructing file:   0%|          |  0.00B /  737MB            

Phase_4_CJPE/Train/train.json: downloading bytes:           |  0.00B            

train | records: 37387 | unique IDs: 32305 | repeated IDs: 5082
Class counts: {0: 22067, 1: 15320}


Phase_4_CJPE/Train/dev.json: reconstructing file:   0%|          |  0.00B / 80.3MB            

Phase_4_CJPE/Train/dev.json: downloading bytes:           |  0.00B            

dev | records: 3505 | unique IDs: 2511 | repeated IDs: 994
Class counts: {0: 1749, 1: 1756}
Selected training records: 37387 → 7463
Subset classes: {0: 4393, 1: 3070}


In [4]:
def chunk_token_ids(tokens, max_len=MAX_SEQ_LEN, overlap=CHUNK_OVERLAP):
    if not 0 <= overlap < max_len: raise ValueError('Invalid chunk overlap')
    if not tokens: return [[EMPTY_ID]]
    chunks = []
    for start in range(0, len(tokens), max_len - overlap):
        chunks.append(tokens[start:start + max_len])
        if start + max_len >= len(tokens): break
    return chunks

def select_chunk_indices(count, limit, training, seed):
    if limit is None or count <= limit:
        return list(range(count))
    if limit < 1: raise ValueError('Chunk limit must be positive or None')
    if training:
        rng = random.Random(seed)
        # One random chunk from each equal-sized region; two regions = document halves.
        return [rng.randrange(i * count // limit, (i + 1) * count // limit)
                for i in range(limit)]
    if limit == 1: return [count // 2]
    return [i * (count - 1) // (limit - 1) for i in range(limit)]

class CJPECaseDataset(Dataset):
    def __init__(self, documents, split):
        self.split, self.epoch = split, 0
        self.rows = []
        token_lengths, chunk_counts = [], []
        for doc in tqdm(documents, desc='Tokenizing cases'):
            tokens = tokenizer.encode(doc['text'], add_special_tokens=False).ids
            chunks = chunk_token_ids(tokens)
            self.rows.append(dict(doc_id=doc['id'], chunks=chunks, label=doc.get('label', -100)))
            token_lengths.append(len(tokens)); chunk_counts.append(len(chunks))
        lengths = torch.tensor(token_lengths, dtype=torch.float)
        print('Cases/chunks:', len(self.rows), sum(chunk_counts), 'max chunks/case:', max(chunk_counts))
        print('Token lengths p50/p90/p99/max:', torch.quantile(lengths, torch.tensor([.5,.9,.99,1.])).tolist())
    def __len__(self): return len(self.rows)
    def set_epoch(self, epoch): self.epoch = epoch
    def __getitem__(self, index):
        row = self.rows[index]
        training = self.split == 'train'
        limit = TRAIN_MAX_CHUNKS if training else EVAL_MAX_CHUNKS
        selected = select_chunk_indices(len(row['chunks']), limit, training,
                                        SEED + self.epoch * len(self.rows) + index)
        return dict(row, chunks=[row['chunks'][i] for i in selected])

def collate_cjpe(rows):
    chunks, owners = [], []
    for case_index, row in enumerate(rows):
        chunks.extend(row['chunks'])
        owners.extend([case_index] * len(row['chunks']))
    order = sorted(range(len(chunks)), key=lambda i: len(chunks[i]), reverse=True)
    chunks = [chunks[i] for i in order]
    owners = [owners[i] for i in order]
    lengths = torch.tensor([len(c) for c in chunks], dtype=torch.long)
    ids = torch.full((len(chunks), int(lengths.max())), PAD_ID, dtype=torch.long)
    for i, chunk in enumerate(chunks): ids[i, :len(chunk)] = torch.tensor(chunk)
    return dict(input_ids=ids, lengths=lengths, chunk_to_case=torch.tensor(owners, dtype=torch.long),
                labels=torch.tensor([r['label'] for r in rows], dtype=torch.long),
                doc_ids=[r['doc_id'] for r in rows])

datasets = {split: CJPECaseDataset(docs, split) for split, docs in data.items()}
def make_loader(split):
    return DataLoader(datasets[split], batch_size=BATCH_SIZE, shuffle=(split == 'train'),
                      collate_fn=collate_cjpe, num_workers=0, pin_memory=(DEVICE == 'cuda'))


Tokenizing cases:   0%|          | 0/7463 [00:00<?, ?it/s]

Cases/chunks: 7463 19057 max chunks/case: 235
Token lengths p50/p90/p99/max: [2661.0, 8258.0, 23486.49609375, 465531.0]


Tokenizing cases:   0%|          | 0/3505 [00:00<?, ?it/s]

Cases/chunks: 3505 10656 max chunks/case: 31
Token lengths p50/p90/p99/max: [3877.0, 9470.19921875, 25065.775390625, 60245.0]


In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F


# ============================================================
# RoPE
# ============================================================

class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim, max_seq_len=2048, base=10000):
        super().__init__()

        inv_freq = 1.0 / (
            base ** (torch.arange(0, head_dim, 2).float() / head_dim)
        )

        positions = torch.arange(max_seq_len).float()
        freqs = torch.outer(positions, inv_freq)

        self.register_buffer("cos", freqs.cos(), persistent=False)
        self.register_buffer("sin", freqs.sin(), persistent=False)

    def forward(self, q, k):
        # q, k: [B, H, T, D]
        T = q.size(2)

        cos = self.cos[:T].to(dtype=q.dtype).unsqueeze(0).unsqueeze(0)
        sin = self.sin[:T].to(dtype=q.dtype).unsqueeze(0).unsqueeze(0)

        q1, q2 = q[..., ::2], q[..., 1::2]
        k1, k2 = k[..., ::2], k[..., 1::2]

        q_rot = torch.stack(
            [q1 * cos - q2 * sin,
             q1 * sin + q2 * cos],
            dim=-1,
        ).flatten(-2)

        k_rot = torch.stack(
            [k1 * cos - k2 * sin,
             k1 * sin + k2 * cos],
            dim=-1,
        ).flatten(-2)

        return q_rot, k_rot


# ============================================================
# SwiGLU
# ============================================================

class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()

        self.gate_proj = nn.Linear(d_model, d_ff, bias=False)
        self.up_proj = nn.Linear(d_model, d_ff, bias=False)
        self.down_proj = nn.Linear(d_ff, d_model, bias=False)

    def forward(self, x):
        return self.down_proj(
            F.silu(self.gate_proj(x)) * self.up_proj(x)
        )


# ============================================================
# GQA + SDPA
# ============================================================

class GQAAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len=2048):
        super().__init__()

        assert d_model % n_heads == 0
        assert n_heads % n_kv_heads == 0

        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.head_dim = d_model // n_heads

        self.q_proj = nn.Linear(
            d_model,
            n_heads * self.head_dim,
            bias=False,
        )

        self.k_proj = nn.Linear(
            d_model,
            n_kv_heads * self.head_dim,
            bias=False,
        )

        self.v_proj = nn.Linear(
            d_model,
            n_kv_heads * self.head_dim,
            bias=False,
        )

        self.out_proj = nn.Linear(
            d_model,
            d_model,
            bias=False,
        )

        self.rope = RotaryEmbedding(
            self.head_dim,
            max_seq_len=max_seq_len,
        )

    def forward(self, x):

        B, T, _ = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(
            B, T,
            self.n_heads,
            self.head_dim
        ).transpose(1, 2)

        k = k.view(
            B, T,
            self.n_kv_heads,
            self.head_dim
        ).transpose(1, 2)

        v = v.view(
            B, T,
            self.n_kv_heads,
            self.head_dim
        ).transpose(1, 2)

        q, k = self.rope(q, k)
        q = q.to(v.dtype)
        k = k.to(v.dtype)

        # Repeat KV heads for GQA
        repeat_factor = self.n_heads // self.n_kv_heads

        k = k.repeat_interleave(
            repeat_factor,
            dim=1,
        )

        v = v.repeat_interleave(
            repeat_factor,
            dim=1,
        )

        # PyTorch SDPA can dispatch to Flash Attention
        # / memory-efficient kernels automatically.
        attn_out = F.scaled_dot_product_attention(
            q,
            k,
            v,
            is_causal=True,
        )

        attn_out = (
            attn_out
            .transpose(1, 2)
            .contiguous()
            .view(B, T, -1)
        )

        return self.out_proj(attn_out)


# ============================================================
# Decoder Block
# ============================================================

class DecoderBlock(nn.Module):
    def __init__(
        self,
        d_model,
        n_heads,
        n_kv_heads,
        d_ff,
        max_seq_len=2048,
    ):
        super().__init__()

        self.norm1 = nn.RMSNorm(d_model)
        self.attention = GQAAttention(
            d_model,
            n_heads,
            n_kv_heads,
            max_seq_len=max_seq_len,
        )

        self.norm2 = nn.RMSNorm(d_model)
        self.ffn = SwiGLU(
            d_model,
            d_ff,
        )

    def forward(self, x):

        x = x + self.attention(
            self.norm1(x)
        )

        x = x + self.ffn(
            self.norm2(x)
        )

        return x


# ============================================================
# Decoder-only Language Model
# ============================================================

class EfficientDecoderLM(nn.Module):

    def __init__(
        self,
        vocab_size,
        d_model=768,
        n_heads=12,
        n_kv_heads=4,
        n_layers=12,
        d_ff=3072,
        max_seq_len=2048,
    ):
        super().__init__()

        self.vocab_size = vocab_size
        self.d_model = d_model

        self.token_embedding = nn.Embedding(                             
            vocab_size,
            d_model,
        )

        self.blocks = nn.ModuleList([
            DecoderBlock(
                d_model=d_model,
                n_heads=n_heads,
                n_kv_heads=n_kv_heads,
                d_ff=d_ff,
                max_seq_len=max_seq_len,
            )
            for _ in range(n_layers)
        ])

        self.final_norm = nn.RMSNorm(d_model)

        # Tied LM head
        self.lm_head = nn.Linear(
            d_model,
            vocab_size,
            bias=False,
        )

        self.lm_head.weight = self.token_embedding.weight

        self.apply(self._init_weights)

    def _init_weights(self, module):

        if isinstance(module, nn.Linear):
            nn.init.xavier_uniform_(module.weight)

        elif isinstance(module, nn.Embedding):
            nn.init.xavier_uniform_(module.weight)

        elif isinstance(module, nn.RMSNorm):
            nn.init.ones_(module.weight)

    def forward(self, input_ids):

        x = self.token_embedding(input_ids)

        for block in self.blocks:
            x = block(x)

        x = self.final_norm(x)

        return self.lm_head(x)


In [6]:
from torch.utils.checkpoint import checkpoint as activation_checkpoint

class EfficientDecoderLMForCJPE(EfficientDecoderLM):
    def __init__(self, vocab_size, head_dropout=0.1, **kwargs):
        super().__init__(vocab_size=vocab_size, **kwargs)
        del self.lm_head
        self.cjpe_dropout = nn.Dropout(head_dropout)
        self.cjpe_head = nn.Linear(self.d_model, 2)
        self.frozen_prefix_blocks = 0
        self.config = {"model_type": "custom_decoder_cjpe"}

    def encode_chunks(self, input_ids, lengths):
        with torch.no_grad():
            x = self.token_embedding(input_ids)
            for i in range(self.frozen_prefix_blocks):
                x = self.blocks[i](x)
        for i in range(self.frozen_prefix_blocks, len(self.blocks)):
            x = self.blocks[i](x)
        x = self.final_norm(x)
        # Right padding cannot affect preceding real tokens under causal attention.
        mask = torch.arange(x.size(1), device=x.device)[None, :] < lengths[:, None]
        return (x * mask.unsqueeze(-1)).sum(1, dtype=torch.float32) / lengths[:, None].float()

    def forward(self, input_ids, lengths, chunk_to_case, num_cases):
        device = self.cjpe_head.weight.device
        pooled_chunks = []
        for start in range(0, input_ids.size(0), CHUNK_ENCODE_BATCH_SIZE):
            end = min(start + CHUNK_ENCODE_BATCH_SIZE, input_ids.size(0))
            lens = lengths[start:end].to(device, non_blocking=True)
            # Trim this microbatch to its own longest chunk.
            width = int(lengths[start:end].max())
            ids = input_ids[start:end, :width].to(device, non_blocking=True)
            if self.training and USE_GRADIENT_CHECKPOINTING:
                h = activation_checkpoint(self.encode_chunks, ids, lens, use_reentrant=False,
                                          preserve_rng_state=True)
            else: h = self.encode_chunks(ids, lens)
            pooled_chunks.append(h)
        h = torch.cat(pooled_chunks, dim=0)
        owners = chunk_to_case.to(device, non_blocking=True)
        sums = h.new_zeros((num_cases, self.d_model)).index_add(0, owners, h)
        counts = torch.bincount(owners, minlength=num_cases).to(h.dtype).unsqueeze(1)
        assert bool((counts > 0).all()), 'Every case must have at least one chunk'
        document_vectors = sums / counts
        return self.cjpe_head(self.cjpe_dropout(document_vectors))

def forward_batch(model, batch):
    with autocast_context():
        return model(batch['input_ids'], batch['lengths'], batch['chunk_to_case'], len(batch['doc_ids']))

from peft import AdaLoraConfig
from peft.tuners.adalora import AdaLoraModel
import peft
import transformers

# Schedule units are successful optimizer updates, not microbatches.
UPDATES_PER_EPOCH = math.ceil(len(make_loader('train')) / GRAD_ACCUM_STEPS)
RANK_TOTAL_STEPS = max(3, UPDATES_PER_EPOCH * min(RANK_SCHEDULE_EPOCHS, NUM_EPOCHS))
RANK_TINIT = max(1, int(RANK_TOTAL_STEPS * RANK_WARMUP_FRACTION))
RANK_TFINAL = max(1, int(RANK_TOTAL_STEPS * RANK_FINAL_FRACTION))
if RANK_TINIT + RANK_TFINAL >= RANK_TOTAL_STEPS:
    raise ValueError('AdaLoRA warmup/final phases leave no budgeting phase')
phase1_path = download_bucket_file(MODEL_PATH)

def build_cjpe_model():
    backbone = EfficientDecoderLMForCJPE(VOCAB_SIZE, head_dropout=HEAD_DROPOUT, **MODEL_CONFIG)
    checkpoint = torch.load(phase1_path, map_location='cpu', weights_only=False)
    if 'model_state_dict' in checkpoint: state = checkpoint['model_state_dict']
    elif 'model' in checkpoint: state = checkpoint['model']
    elif 'state_dict' in checkpoint: state = checkpoint['state_dict']
    elif all(isinstance(v, torch.Tensor) for v in checkpoint.values()): state = checkpoint
    else: raise ValueError(f'Cannot identify model weights: {list(checkpoint)}')
    state = dict(state)
    state.pop('lm_head.weight', None)
    missing, unexpected = backbone.load_state_dict(state, strict=False)
    assert set(missing) == {'cjpe_head.weight', 'cjpe_head.bias'} and not unexpected, (missing, unexpected)
    del state, checkpoint
    for p in backbone.parameters(): p.requires_grad = False
    if not 1 <= ADAPT_LAST_N_BLOCKS <= len(backbone.blocks):
        raise ValueError('Invalid ADAPT_LAST_N_BLOCKS')
    first = len(backbone.blocks) - ADAPT_LAST_N_BLOCKS
    targets = [f'blocks.{i}.attention.{name}'
               for i in range(first, len(backbone.blocks))
               for name in ['q_proj', 'k_proj', 'v_proj', 'out_proj']]
    config = AdaLoraConfig(
        **ADAPTER_CONFIG, target_modules=targets, bias='none',
        total_step=RANK_TOTAL_STEPS, tinit=RANK_TINIT, tfinal=RANK_TFINAL,
        inference_mode=False)
    # Direct tuner supports this plain nn.Module without a Transformers LM wrapper.
    model = AdaLoraModel(backbone, config, adapter_name='default')
    if UNFREEZE_LAST_N_BLOCKS:
        for block in backbone.blocks[-UNFREEZE_LAST_N_BLOCKS:]:
            for name, p in block.named_parameters():
                if 'ranknum' not in name: p.requires_grad = True
    for p in backbone.cjpe_head.parameters(): p.requires_grad = True
    backbone.frozen_prefix_blocks = next(
        (i for i, block in enumerate(backbone.blocks)
         if any(p.requires_grad for p in block.parameters())), len(backbone.blocks))
    model = model.to(DEVICE)
    print('AdaLoRA targets:', targets)
    print('Frozen prefix blocks:', backbone.frozen_prefix_blocks)
    print('Rank schedule updates / warmup / final:', RANK_TOTAL_STEPS, RANK_TINIT, RANK_TFINAL)
    print('Total/trainable parameters:', sum(p.numel() for p in model.parameters()),
          sum(p.numel() for p in model.parameters() if p.requires_grad))
    return model

def orthogonality_loss(model):
    # PEFT's direct custom-model tuner does not add this to our CE loss.
    terms = []
    with torch.autocast(device_type=DEVICE, enabled=False):
        for name, p in model.named_parameters():
            if not p.requires_grad: continue
            if 'lora_A.' in name:
                gram = p.float() @ p.float().T
            elif 'lora_B.' in name:
                gram = p.float().T @ p.float()
            else: continue
            terms.append(torch.linalg.matrix_norm(
                gram - torch.eye(gram.size(0), device=gram.device), ord='fro'))
    return torch.stack(terms).mean()

def allocation_state(model):
    allocator = model.rankallocator
    return dict(rank_pattern=model.peft_config['default'].rank_pattern,
                ema={key: {name: value.detach().cpu().clone() for name, value in getattr(allocator, key).items()}
                     for key in ['ipt', 'exp_avg_ipt', 'exp_avg_unc']})

def restore_allocation_state(model, state):
    model.peft_config['default'].rank_pattern = state['rank_pattern']
    device = next(model.parameters()).device
    for key, values in state['ema'].items():
        setattr(model.rankallocator, key, {name: value.to(device) for name, value in values.items()})

def rank_report(model):
    pattern = model.peft_config['default'].rank_pattern
    if pattern is None:
        return {'initial_rank_per_projection': ADAPTER_CONFIG['init_r']}
    return {name: sum(bool(x) for x in mask) for name, mask in pattern.items()}


hf://buckets/A-240031/legal-lm-phase1-bu(…): reconstructing file:   0%|          |  0.00B /  515MB            

hf://buckets/A-240031/legal-lm-phase1-bu(…): downloading bytes:           |  0.00B            

In [7]:
def experiment_metadata(domain='CJPE'):
    return dict(domain=domain, phase1_tag=PHASE1_TAG, model_path=MODEL_PATH, tokenizer_path=TOKENIZER_PATH,
                labels=LABELS, model_config=MODEL_CONFIG, adapter_config=ADAPTER_CONFIG,
                train_fraction=TRAIN_FRACTION, subset_digest=SUBSET_DIGEST, data_digest=DATA_DIGEST,
                adapt_last_n_blocks=ADAPT_LAST_N_BLOCKS,
                rank_total_steps=RANK_TOTAL_STEPS, rank_tinit=RANK_TINIT, rank_tfinal=RANK_TFINAL,
                rank_schedule_epochs=RANK_SCHEDULE_EPOCHS,
                peft_version=peft.__version__, transformers_version=transformers.__version__,
                max_seq_len=MAX_SEQ_LEN, chunk_overlap=CHUNK_OVERLAP,
                chunk_encode_batch_size=CHUNK_ENCODE_BATCH_SIZE,
                gradient_checkpointing=USE_GRADIENT_CHECKPOINTING,
                train_max_chunks=TRAIN_MAX_CHUNKS, eval_max_chunks=EVAL_MAX_CHUNKS,
                sampling='stratified_epoch_seed_v1', use_amp=USE_AMP, amp_dtype=str(AMP_DTYPE),
                unfreeze_last_n_blocks=UNFREEZE_LAST_N_BLOCKS,
                pooling='token_mean_then_chunk_mean', input_format='overlapping_chunks_v1',
                data_repo=DATA_REPO, data_prefix=DATA_PREFIX, label_key=LABEL_KEY,
                technique='AdaLoRA_PEFT', head_dropout=HEAD_DROPOUT,
                seed=SEED, batch_size=BATCH_SIZE, grad_accum_steps=GRAD_ACCUM_STEPS,
                lr=LR, weight_decay=WEIGHT_DECAY, grad_clip=GRAD_CLIP,
                patience=PATIENCE, min_delta=MIN_DELTA)

def save_cjpe_checkpoint(tag, model, optimizer, epoch, extra, domain):
    folder = WORK_DIR / 'checkpoints' / tag
    folder.mkdir(parents=True, exist_ok=True)
    rng = {'python': random.getstate(), 'torch': torch.get_rng_state()}
    if torch.cuda.is_available(): rng['cuda'] = torch.cuda.get_rng_state_all()
    payload = dict(model=model.state_dict(), optimizer=optimizer.state_dict(), rng=rng,
                   epoch=epoch, extra=extra, adalora=allocation_state(model), metadata=experiment_metadata(domain))
    tmp = folder / 'checkpoint.tmp'
    torch.save(payload, tmp)
    os.replace(tmp, folder / 'checkpoint.pt')
    if UPLOAD_CHECKPOINTS:
        fs.put_file(str(folder / 'checkpoint.pt'), checkpoint_remote_path(tag))

def load_cjpe_checkpoint(tag, model, domain, optimizer=None, restore_rng=False):
    path = WORK_DIR / 'checkpoints' / tag / 'checkpoint.pt'
    if not path.exists() or restore_rng:
        if UPLOAD_CHECKPOINTS:
            path = download_bucket_file(checkpoint_remote_path(tag))
        elif not path.exists():
            raise FileNotFoundError(path)
    payload = torch.load(path, map_location='cpu', weights_only=False)  # own trusted checkpoints only
    assert payload['metadata'] == experiment_metadata(domain), 'Checkpoint configuration mismatch'
    model.load_state_dict(payload['model'])
    restore_allocation_state(model, payload['adalora'])
    if optimizer is not None: optimizer.load_state_dict(payload['optimizer'])
    if restore_rng:
        random.setstate(payload['rng']['python'])
        torch.set_rng_state(payload['rng']['torch'])
        if torch.cuda.is_available() and 'cuda' in payload['rng']:
            torch.cuda.set_rng_state_all(payload['rng']['cuda'])
    return payload

@torch.no_grad()
def evaluate_cjpe(model, loader):
    model.eval()
    targets, predictions = [], []
    total_loss = total_n = 0
    for batch in loader:
        y = batch['labels'].to(DEVICE)
        logits = forward_batch(model, batch)
        total_loss += F.cross_entropy(logits.float(), y, reduction='sum').item()
        total_n += len(y)
        targets.extend(y.cpu().tolist())
        predictions.extend(logits.argmax(-1).cpu().tolist())
    assert total_n > 0, 'Development set has no cases'
    p, r, f, support = precision_recall_fscore_support(targets, predictions, labels=[0, 1], zero_division=0)
    return dict(loss=total_loss / total_n, macro_f1=float(f.mean()), macro_precision=float(p.mean()),
                macro_recall=float(r.mean()), per_label={label: dict(precision=float(p[i]), recall=float(r[i]),
                f1=float(f[i]), support=int(support[i])) for i, label in enumerate(LABELS)},
                confusion_matrix=confusion_matrix(targets, predictions, labels=[0, 1]).tolist())


In [8]:
def train_cjpe(domain='CJPE'):
    assert domain == 'CJPE'
    assert_run_destination()
    seed_everything(SEED)
    model = build_cjpe_model()
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=LR, weight_decay=WEIGHT_DECAY)
    scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP and AMP_DTYPE == torch.float16)
    tag = SAVE_TAG
    best_tag = BEST_TAG
    start = 0
    extra = dict(best_f1=-1.0, patience_reference=-1.0, stale=0, history=[], best_tag=best_tag, global_step=0)
    if RESUME:
        checkpoint = load_cjpe_checkpoint(SAVE_TAG, model, domain, optimizer, restore_rng=True)
        start, extra = checkpoint['epoch'], checkpoint['extra']
        best_tag = extra['best_tag']
        if 'scaler' in extra: scaler.load_state_dict(extra['scaler'])
        del checkpoint
    train_loader, dev_loader = make_loader('train'), make_loader('dev')
    assert len(train_loader) > 0
    for epoch in range(start, NUM_EPOCHS):
        if extra['stale'] >= PATIENCE: break
        datasets['train'].set_epoch(epoch)
        model.train()
        optimizer.zero_grad(set_to_none=True)
        train_sum = train_n = group_n = 0
        for step, batch in enumerate(tqdm(train_loader, desc=f'{domain} epoch {epoch+1}')):
            y = batch['labels'].to(DEVICE)
            logits = forward_batch(model, batch)
            ce_sum = F.cross_entropy(logits.float(), y, reduction='sum')
            reg = orthogonality_loss(model)
            loss_sum = ce_sum + len(y) * ADAPTER_CONFIG['orth_reg_weight'] * reg
            scaler.scale(loss_sum).backward()
            group_n += len(y)
            train_n += len(y)
            train_sum += ce_sum.item()
            if (step + 1) % GRAD_ACCUM_STEPS == 0 or step + 1 == len(train_loader):
                scaler.unscale_(optimizer)
                for p in params:
                    if p.grad is not None: p.grad.div_(group_n)
                torch.nn.utils.clip_grad_norm_(params, GRAD_CLIP)
                previous_scale = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()
                successful_update = (not scaler.is_enabled() or scaler.get_scale() >= previous_scale)
                if successful_update:
                    extra['global_step'] += 1
                    # Gradients must still be present here for importance scoring.
                    model.update_and_allocate(extra['global_step'])
                optimizer.zero_grad(set_to_none=True)
                group_n = 0
        metrics = evaluate_cjpe(model, dev_loader)
        f1 = metrics['macro_f1']
        improved = f1 > extra['best_f1']
        if improved: extra['best_f1'] = f1
        if f1 > extra['patience_reference'] + MIN_DELTA:
            extra['patience_reference'], extra['stale'] = f1, 0
        else: extra['stale'] += 1
        extra['history'].append(dict(epoch=epoch+1, train_loss=train_sum/train_n,
                                     global_step=extra['global_step'], ranks=rank_report(model), **metrics))
        print('AdaLoRA ranks:', rank_report(model))
        print(f"CJPE train loss={train_sum/train_n:.4f}, dev macro-F1={f1:.4f}, precision={metrics['macro_precision']:.4f}, recall={metrics['macro_recall']:.4f}, patience={extra['stale']}/{PATIENCE}")
        extra['scaler'] = scaler.state_dict()
        if improved: save_cjpe_checkpoint(best_tag, model, optimizer, epoch+1, extra, domain)
        save_cjpe_checkpoint(tag, model, optimizer, epoch+1, extra, domain)
    load_cjpe_checkpoint(best_tag, model, domain)
    metrics = evaluate_cjpe(model, dev_loader)
    with open(WORK_DIR / f'{domain}_dev_metrics.json', 'w') as f: json.dump(metrics, f, indent=2)
    with open(WORK_DIR / f'{domain}_history.json', 'w') as f: json.dump(extra['history'], f, indent=2)
    print('Best development metrics:', metrics)
    del optimizer
    return model, extra['history']


In [9]:
def validate_predictions(predictions, documents):
    assert len(predictions) == len(documents)
    assert [p['id'] for p in predictions] == [d['id'] for d in documents]
    assert len({p['id'] for p in predictions}) == len(predictions)
    assert all(set(p) == {'id', 'pred'} and type(p['pred']) is int and p['pred'] in (0, 1) for p in predictions)

@torch.no_grad()
def predict_cjpe(model):
    model.eval()
    predictions = []
    for batch in tqdm(make_loader('test'), desc='CJPE test predictions'):
        labels = forward_batch(model, batch).argmax(-1).cpu().tolist()
        predictions.extend(dict(id=doc_id, pred=int(label)) for doc_id, label in zip(batch['doc_ids'], labels))
    validate_predictions(predictions, data['test'])
    return predictions

cjpe_model, cjpe_history = train_cjpe()
# Load test only after training. Its IDs are unique in the supplied data.
data['test'] = normalize_documents(
    download_json(f'{DATA_PREFIX}/Test/test.json'), labelled=False)
datasets['test'] = CJPECaseDataset(data['test'], 'test')
predictions = predict_cjpe(cjpe_model)
json_path = WORK_DIR / 'prediction.json'
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)
del cjpe_model
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()


AdaLoRA targets: ['blocks.6.attention.q_proj', 'blocks.6.attention.k_proj', 'blocks.6.attention.v_proj', 'blocks.6.attention.out_proj', 'blocks.7.attention.q_proj', 'blocks.7.attention.k_proj', 'blocks.7.attention.v_proj', 'blocks.7.attention.out_proj']
Frozen prefix blocks: 6
Rank schedule updates / warmup / final: 1401 140 280
Total/trainable parameters: 128891018 231554


CJPE epoch 1:   0%|          | 0/7463 [00:00<?, ?it/s]

AdaLoRA ranks: {'blocks.6.attention.q_proj.lora_E.default': 7, 'blocks.6.attention.k_proj.lora_E.default': 14, 'blocks.6.attention.v_proj.lora_E.default': 14, 'blocks.6.attention.out_proj.lora_E.default': 6, 'blocks.7.attention.q_proj.lora_E.default': 6, 'blocks.7.attention.k_proj.lora_E.default': 14, 'blocks.7.attention.v_proj.lora_E.default': 15, 'blocks.7.attention.out_proj.lora_E.default': 7}
CJPE train loss=0.6744, dev macro-F1=0.5182, precision=0.5636, recall=0.5480, patience=0/5


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

... (38 outputs left out)

Phase_4_CJPE/Test/test.json: downloading bytes:           |  0.00B            

Tokenizing cases:   0%|          | 0/1517 [00:00<?, ?it/s]

Cases/chunks: 1517 4664 max chunks/case: 29
Token lengths p50/p90/p99/max: [3967.0, 9680.0, 24404.427734375, 56610.0]


CJPE test predictions:   0%|          | 0/1517 [00:00<?, ?it/s]

In [10]:
# Packaging can be rerun without retraining.
json_path = WORK_DIR / 'prediction.json'
with open(json_path, encoding='utf-8') as f: predictions = json.load(f)
validate_predictions(predictions, data['test'])
zip_path = WORK_DIR / SUBMISSION_ZIP_NAME
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    z.write(json_path, arcname='prediction.json')
print('JSON:', json_path)
print('ZIP:', zip_path)
print('Cases:', len(predictions))


JSON: /kaggle/working/rr_runs/cjpe_decoder_126M_adalora_len2048_ov64_sample2_eval4_data20_last2/prediction.json
ZIP: /kaggle/working/rr_runs/cjpe_decoder_126M_adalora_len2048_ov64_sample2_eval4_data20_last2/cjpe_decoder_126M_adalora_len2048_ov64_sample2_eval4_data20_last2.zip
Cases: 1517
